In [1]:
import os
api_key = os.environ['AZURE_OPENAI_API_KEY']
base_url = os.environ['AZURE_OPENAI_ENDPOINT']
base_url = f"{base_url}/openai/v1"
from openai import OpenAI
client = OpenAI(api_key=api_key, base_url=base_url)

In [2]:
import requests,json
def get_current_weather(city:str)->dict:

    api_key = "6a8b0ac166a37e2b7a38e64416b3c3fe"

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = response.content.decode()
    response = json.loads(response)
    output = {"City Name":city,"weather":response["weather"][0]['description'],
              "temperature":response['main']['temp'],
              "unit":"kelvin"}

    return output

In [3]:
get_current_weather("delhi")

{'City Name': 'delhi',
 'weather': 'scattered clouds',
 'temperature': 310.48,
 'unit': 'kelvin'}

In [4]:
get_current_weather("mumbai")

{'City Name': 'mumbai',
 'weather': 'clear sky',
 'temperature': 303.97,
 'unit': 'kelvin'}

In [5]:
tools = [{"type":"function",
          "name":"get_current_weather",
          "description":"This function can be used to fetch current weather information for any given city or location.",
          "parameters":{"type":"object",
                        "properties":{"city":{"type":"string","description":"name of location or city, e.g. mumbai, new york"},
                                      },
                        },
                        "required":['city'],
},
]

In [14]:
tool_map = {"get_current_weather":get_current_weather}

def generate_resp(prompt,model='gpt-4o-mini'):
    messages = [{"role":"user",'content':prompt}]

    while True:
        response = client.responses.create(model=model,input=messages,
                                           instructions="you are helpful assistant",
                                       tools=tools)
        response_op = response.output
        function_calls = [item for item in response_op if item.type=="function_call"]
        if function_calls:
            print(response_op)
            messages = messages + response_op

            for tool in function_calls:
                tool_name = tool.name
                tool_args = json.loads(tool.arguments)
                tool_func = tool_map[tool_name]
                tool_output = tool_func(**tool_args)
                messages = messages + [{"type":"function_call_output","call_id":tool.call_id,
                                        "output":json.dumps(tool_output)}]
        else:
            break
    return response.output_text
        
        
   

In [15]:
print(generate_resp("what is AI?"))

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are designed to think and learn like humans. It encompasses various technologies and methodologies that enable computers to perform tasks that traditionally require human intelligence, such as:

1. **Learning**: Acquiring new information and rules for using it.
2. **Reasoning**: Using rules to reach approximate or definite conclusions.
3. **Problem-Solving**: Finding solutions to complex problems.
4. **Perception**: Interpreting sensory data (e.g., visual, auditory).
5. **Natural Language Processing (NLP)**: Understanding and interacting with human language.

AI can be classified into two main types:

- **Narrow AI**: Designed to perform specific tasks (e.g., facial recognition, recommendation systems).
- **General AI**: Hypothetical, capable of performing any intellectual task a human can do.

AI technologies are used in various fields, including healthcare, finance, transportation, and entert

In [16]:
generate_resp("what is the weather today in manali?")

[ResponseFunctionToolCall(arguments='{"city":"Manali"}', call_id='call_RXuKHtpuRVx34qBMimBx8ZrL', name='get_current_weather', type='function_call', id='fc_0ca43ea24867af450069aeab734f488197b1785c5d82e6b356', status='completed')]


'Today in Manali, the weather is clear with a temperature of approximately 300.75 K, which is about 27.6°C.'

In [18]:
print(generate_resp("what is the weather today in manali and delhi?"))

[ResponseFunctionToolCall(arguments='{"city":"Manali"}', call_id='call_XPHdpFRDBb3X26glA3fcj4Vm', name='get_current_weather', type='function_call', id='fc_04b451ab70846ae70069aeabc91dd88193bc1c889843ed5f3d', status='completed'), ResponseFunctionToolCall(arguments='{"city":"Delhi"}', call_id='call_A3DdSza6UXV1EWJRpB9WT01F', name='get_current_weather', type='function_call', id='fc_04b451ab70846ae70069aeabc91de88193b31f0b2a381a991e', status='completed')]
### Today's Weather

**Manali:**
- **Condition:** Clear sky 
- **Temperature:** 27.6°C (300.75 K)

**Delhi:**
- **Condition:** Scattered clouds 
- **Temperature:** 37.3°C (310.48 K)

If you need more details, feel free to ask!


In [20]:
print(generate_resp("What is capital city of India and What is weather there?"))

[ResponseFunctionToolCall(arguments='{"city":"New Delhi"}', call_id='call_UhqF7454hxXBcnhmn0ix2iSU', name='get_current_weather', type='function_call', id='fc_0481da2b9751cc8f0069aeac1ef36481969ae1af08e806d6c5', status='completed')]
The capital city of India is **New Delhi**. Currently, the weather there is characterized by **scattered clouds** with a temperature of approximately **310.37 K** (about 37.2 °C).
